# ASA Conecta — Entrega 1 de IA e Aprendizado de Máquina
**Baseline do Agente para o Estudante / Monitoramento de risco**

Objetivos: analisar a base unificada, preparar atributos, construir uma baseline explicável, definir métricas, criar score inicial, tabela de decisão/priorização e um RAG inicial com avaliação de relevância e respostas com fontes.

> O modelo é apoio à decisão e à priorização. Não executa sanções nem decisões administrativas.

In [ ]:
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score, confusion_matrix
from sklearn.inspection import permutation_importance
import matplotlib.pyplot as plt

## 1. Dados e rótulo
A base final possui uma linha por aluno/registro consolidado e 24 colunas. `evadiu` é o alvo. `ID_ALUNO` é identificador e não entra no modelo. O status acadêmico que originou o rótulo também não entra como feature, evitando vazamento.

In [ ]:
df = pd.read_csv("base_unificada_asa-final.csv").dropna(subset=["evadiu"]).copy()
df["evadiu"] = df["evadiu"].astype(int)
display(df.head())
print(df.shape)
display(df["evadiu"].value_counts().rename("n"))
display((df.isna().mean()*100).sort_values(ascending=False).head(10).rename("% nulos"))

## 2. Features e preparação
Numéricas: imputação pela mediana + padronização. Categóricas: imputação pela moda + one-hot encoding. A Regressão Logística foi escolhida como baseline por ser simples, reproduzível e explicável.

In [ ]:
features = [c for c in df.columns if c not in ["ID_ALUNO","evadiu"]]
cat = [c for c in features if df[c].dtype == "object"]
num = [c for c in features if c not in cat]
X, y = df[features], df["evadiu"]

pre = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")),("scaler", StandardScaler())]), num),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),("onehot", OneHotEncoder(handle_unknown="ignore"))]), cat)
])
model = Pipeline([("preprocess", pre),("model", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42))])

## 3. Avaliação
Como a evasão é minoritária, não usamos apenas acurácia. Métricas principais: **ROC-AUC**, **PR-AUC**, **Recall da evasão**, **Precision** e **F1**. O `class_weight='balanced'` reduz a tendência de ignorar a classe minoritária.

In [ ]:
Xtr,Xte,ytr,yte = train_test_split(X,y,test_size=.20,stratify=y,random_state=42)
model.fit(Xtr,ytr)
p = model.predict_proba(Xte)[:,1]
pred = (p>=.50).astype(int)
print(classification_report(yte,pred,digits=3))
print("ROC-AUC:", roc_auc_score(yte,p))
print("PR-AUC:", average_precision_score(yte,p))
print("Matriz de confusão:\n", confusion_matrix(yte,pred))

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cvres = cross_validate(model, X, y, cv=cv, scoring=["roc_auc","average_precision","f1","recall"])
for k,v in cvres.items():
    if k.startswith("test_"): print(k, round(v.mean(),4), "+/-", round(v.std(),4))

## 4. Explicação dos fatores
Usamos importância por permutação no conjunto de teste: mede quanto o ROC-AUC cai quando um atributo é embaralhado. Isso é uma explicação global de associação preditiva, **não causalidade**.

In [ ]:
imp = permutation_importance(model,Xte,yte,n_repeats=5,random_state=42,scoring="roc_auc")
importance = pd.Series(imp.importances_mean,index=features).sort_values(ascending=False)
display(importance.head(12))

## 5. Score e faixas iniciais
Primeira regra operacional: Verde `< 0,35`; Amarelo `0,35–0,65`; Vermelho `>= 0,65`. Esses limites são **iniciais** e precisam ser calibrados com custo de falso positivo/falso negativo e validação do ASA.

In [ ]:
scores = pd.DataFrame({"real":yte.values,"score":p})
scores["faixa"] = pd.cut(scores.score,[-np.inf,.35,.65,np.inf],labels=["Verde","Amarelo","Vermelho"])
display(scores.groupby("faixa", observed=False).agg(n=("real","size"), taxa_evasao=("real","mean"), score_medio=("score","mean")))

## 6. Pendências — regras e priorização
A tabela `tabela_decisao_pendencias.csv` combina score e sinais acadêmicos/financeiros. Nenhuma regra deve bloquear matrícula ou aplicar sanção automaticamente; a saída é uma fila de revisão/acolhimento.

## 7. RAG inicial
Baseline de recuperação: TF-IDF sobre trechos do guia/dicionário. Avaliação inicial usa **Hit@3**: se a fonte esperada aparece entre os 3 trechos mais relevantes. A resposta deve sempre listar as fontes recuperadas. Para produção, substituir o corpus resumido por documentos oficiais versionados e adicionar embeddings/reranking.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

docs = [
 ("Guia ASA - Rótulo","Status Acadêmico define o rótulo; a coluna usada no rótulo nunca deve virar feature."),
 ("Guia ASA - Financeiro","Atributos financeiros incluem parcelas em aberto, acordos, atraso médio, bolsa e valor total."),
 ("Guia ASA - Histórico","Atributos acadêmicos incluem notas, assiduidade, reprovação, tendências e total de semestres."),
 ("Guia ASA - Relacionamentos","Excluir Motivo da Evasão e sinais explícitos de risco quando causarem vazamento.")
]
v=TfidfVectorizer(strip_accents="unicode",ngram_range=(1,2))
M=v.fit_transform([x[1] for x in docs])
def recuperar(q,k=3):
    s=cosine_similarity(v.transform([q]),M)[0]
    ids=np.argsort(s)[::-1][:k]
    return [(docs[i][0],docs[i][1],float(s[i])) for i in ids]
recuperar("Quais sinais financeiros entram no monitoramento?")

## 8. Limitações e próximos passos
- Validar se a base contém duplicidades temporais e evitar que registros do mesmo aluno apareçam em treino e teste se houver múltiplas linhas por pessoa.
- Fazer split temporal quando houver datas confiáveis, aproximando o uso real.
- Comparar Árvore, Random Forest e KNN sem escolher apenas pela acurácia.
- Calibrar probabilidades e thresholds com o ASA.
- Auditar desempenho por subgrupos, estabilidade e drift.
- No RAG, usar documentos oficiais completos, versionamento, avaliação maior e geração com citações verificáveis.